# 04 - Gold Layer
## 02 - Gold Validation

### Goal
- validate gold tables using SQL
- row counts, null checks, negative checks, distribution checks, summary checks

In [0]:
%run ../01_setup/00_config

In [0]:
import logging
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

## Row count checks

In [0]:
for table in [gold_regional_operations_table, gold_market_volatility_table]:
    try:
        count = spark.table(table).count()
        if count > 0:
            log.info(f"[ROWS]    {table} has {count} rows.")
        else:
            raise ValueError(f"[ROWS]    {table} is empty.")
    except AnalysisException:
        raise ValueError(f"[MISSING] Table not found: {table}")

## Null checks

In [0]:
spark.sql(f"""
SELECT
    'gold_regional_operations' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)     AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)         AS null_region,
    SUM(CASE WHEN avg_price_eur_mwh IS NULL THEN 1 ELSE 0 END) AS null_avg_price,
    SUM(CASE WHEN operational_risk IS NULL THEN 1 ELSE 0 END) AS null_operational_risk
FROM {gold_regional_operations_table}
""").show()

spark.sql(f"""
SELECT
    'gold_market_volatility' AS table_name,
    SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END)     AS null_event_date,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)         AS null_region,
    SUM(CASE WHEN avg_price_eur_mwh IS NULL THEN 1 ELSE 0 END) AS null_avg_price,
    SUM(CASE WHEN volatility_band IS NULL THEN 1 ELSE 0 END) AS null_volatility_band
FROM {gold_market_volatility_table}
""").show()

## Negative value checks

In [0]:
spark.sql(f"""
SELECT 'avg_price_eur_mwh'     AS column_name, COUNT(*) AS negative_count FROM {gold_regional_operations_table} WHERE avg_price_eur_mwh < 0
UNION ALL
SELECT 'total_volume_mwh',               COUNT(*) FROM {gold_regional_operations_table} WHERE total_volume_mwh < 0
UNION ALL
SELECT 'critical_event_count',           COUNT(*) FROM {gold_regional_operations_table} WHERE critical_event_count < 0
UNION ALL
SELECT 'weather_impact_score',           COUNT(*) FROM {gold_regional_operations_table} WHERE weather_impact_score < 0
""").show()

## Distribution checks

In [0]:
spark.sql(f"""
SELECT operational_risk, COUNT(*) AS count
FROM {gold_regional_operations_table}
GROUP BY operational_risk
ORDER BY count DESC
""").show()

spark.sql(f"""
SELECT volatility_band, COUNT(*) AS count
FROM {gold_market_volatility_table}
GROUP BY volatility_band
ORDER BY count DESC
""").show()

## Summary check - cross table consistency

In [0]:
ops_count = spark.table(gold_regional_operations_table).count()
vol_count = spark.table(gold_market_volatility_table).count()

print(f"[SUMMARY] {gold_regional_operations_table}: {ops_count} rows")
print(f"[SUMMARY] {gold_market_volatility_table}:   {vol_count} rows")

if ops_count != vol_count:
    log.warning(f"[SUMMARY] Row count mismatch: {ops_count} vs {vol_count}")
else:
    log.info("[SUMMARY] Row counts consistent across gold tables.")

## Retrieve taskValue from gold outputs task

In [0]:
try:
    expected_count = dbutils.jobs.taskValues.get(taskKey="gold_outputs", key="gold_row_count")
    actual_count   = str(ops_count)
    print(f"Expected rows from gold_outputs task: {expected_count}")
    print(f"Actual rows in gold table:            {actual_count}")
    if expected_count != actual_count:
        log.warning(f"[TASKVALUE] Row count mismatch: expected {expected_count}, got {actual_count}")
    else:
        log.info("[TASKVALUE] Row count matches.")
except Exception:
    log.info("[TASKVALUE] Not running in a job context - skipping taskValues check.")